# 07 - Clean and Align

## Purpose
Fix all issues identified in `06_assess_data.ipynb` and produce clean,
consistently formatted versions of all datasets, ready for database loading.
No analysis. Cleaning and standardization only.

## Inputs
- `data/processed/cbam_defaults.csv`
- `data/processed/eu_import_trade_flows.csv`
- `data/processed/country_grid_electricity.csv`
- `data/processed/hydrogen_route_intensities.csv`
- `data/processed/steel_route_intensity.csv`

## Outputs
- `data/clean/country_crosswalk.csv`
- `data/clean/cbam_defaults_clean.csv`
- `data/clean/eu_import_trade_flows_clean.csv`
- `data/clean/country_grid_electricity_clean.csv`
- `data/clean/hydrogen_route_intensities_clean.csv`
- `data/clean/steel_route_intensity_clean.csv`

## Notes
- Cleaned files are saved to `data/clean/`, preserving originals in
  `data/processed/` unchanged.
- All country names standardized to common English, ASCII only.
- All changes documented in comments and observations cells.
- Issues addressed follow the numbered list in `06_assess_data.ipynb`
  section 7.

In [1]:
# Imports and setup — load all processed datasets and confirm shapes
# match what was assessed in 06_assess_data.ipynb
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.4f}".format)

processed = Path("../data/processed")
clean     = Path("../data/clean")
clean.mkdir(exist_ok=True)

defaults  = pd.read_csv(processed / "cbam_defaults.csv")
flows     = pd.read_csv(processed / "eu_import_trade_flows.csv")
grid      = pd.read_csv(processed / "country_grid_electricity.csv")
hydrogen  = pd.read_csv(processed / "hydrogen_route_intensities.csv")
steel     = pd.read_csv(processed / "steel_route_intensity.csv")

datasets = {
    "cbam_defaults":              defaults,
    "eu_import_trade_flows":      flows,
    "country_grid_electricity":   grid,
    "hydrogen_route_intensities": hydrogen,
    "steel_route_intensity":      steel,
}

for name, df in datasets.items():
    print(f"{name}: {df.shape}")

cbam_defaults: (10671, 10)
eu_import_trade_flows: (165182, 9)
country_grid_electricity: (193936, 10)
hydrogen_route_intensities: (6, 5)
steel_route_intensity: (12, 4)


## 1. Country Crosswalk

Builds a mapping table between all country identifier formats used across
the datasets: ISO2 (trade flows), ISO3 (Ember), and full country names
(CBAM defaults and Ember). The crosswalk is the foundation for all
country-level joins in the database.

Construction steps:
1. Seed from Ember — provides ISO3 and a full name for 215 countries
2. Add ISO2 via pycountry lookup on ISO3
3. Add canonical common English name (ASCII only)
4. Map CBAM default country name variants
5. Apply manual fixes for edge cases pycountry cannot resolve

In [3]:
# Seed the crosswalk from Ember's country list, which already provides
# both a full name and ISO3 code for 215 countries. Then use pycountry
# to look up the ISO2 code for each ISO3, flagging any that don't resolve.
import pycountry

# Extract distinct country/ISO3 pairs from Ember
ember_countries = (grid[["Area", "ISO 3 code"]]
                   .drop_duplicates()
                   .rename(columns={"Area": "name_ember", "ISO 3 code": "iso3"}))

# Look up ISO2 for each ISO3 code via pycountry
def get_iso2(iso3):
    try:
        return pycountry.countries.get(alpha_3=iso3).alpha_2
    except AttributeError:
        return None

ember_countries["iso2"] = ember_countries["iso3"].apply(get_iso2)

# Flag any ISO3 codes pycountry could not resolve
unresolved = ember_countries[ember_countries["iso2"].isnull()]
print(f"ISO3 codes pycountry could not resolve to ISO2: {len(unresolved)}")
print(unresolved[["name_ember", "iso3"]].to_string(index=False))

ISO3 codes pycountry could not resolve to ISO2: 1
name_ember iso3
    Kosovo  XKX


In [4]:
# Add the canonical common English country name using pycountry.
# This becomes the single 'country' column used across all clean datasets.
# We use the common_name if available (e.g. "Bolivia" instead of
# "Bolivia, Plurinational State of"), falling back to the official name.

def get_common_name(iso3):
    try:
        c = pycountry.countries.get(alpha_3=iso3)
        return getattr(c, "common_name", None) or c.name
    except AttributeError:
        return None

ember_countries["country"] = ember_countries["iso3"].apply(get_common_name)

# Flag any that didn't resolve — these need manual entries
unresolved_names = ember_countries[ember_countries["country"].isnull()]
print(f"ISO3 codes with no name resolved: {len(unresolved_names)}")
print(unresolved_names[["name_ember", "iso3"]].to_string(index=False))

print(f"\nSample of resolved names (check common_name is working as expected):")
print(ember_countries[["name_ember", "iso3", "iso2", "country"]].head(20).to_string(index=False))

ISO3 codes with no name resolved: 1
name_ember iso3
    Kosovo  XKX

Sample of resolved names (check common_name is working as expected):
         name_ember iso3 iso2             country
        Afghanistan  AFG   AF         Afghanistan
            Albania  ALB   AL             Albania
            Algeria  DZA   DZ             Algeria
     American Samoa  ASM   AS      American Samoa
             Angola  AGO   AO              Angola
Antigua and Barbuda  ATG   AG Antigua and Barbuda
          Argentina  ARG   AR           Argentina
            Armenia  ARM   AM             Armenia
              Aruba  ABW   AW               Aruba
          Australia  AUS   AU           Australia
            Austria  AUT   AT             Austria
         Azerbaijan  AZE   AZ          Azerbaijan
      Bahamas (the)  BHS   BS             Bahamas
            Bahrain  BHR   BH             Bahrain
         Bangladesh  BGD   BD          Bangladesh
           Barbados  BRB   BB            Barbados
            

In [7]:
# Manual fixes for countries pycountry cannot resolve from ISO3,
# and corrections for Ember's UN-style formal names that won't match
# common English usage. Kosovo is the only pycountry gap; the rest
# are Ember naming conventions we want to normalize.

manual_fixes = {
    # pycountry gaps
    "XKX": {"iso2": "XK", "country": "Kosovo"},

    # Ember UN-style formal names -> common English
    "BHS": {"country": "Bahamas"},
    "CPV": {"country": "Cape Verde"},
    "CIV": {"country": "Ivory Coast"},
    "REU": {"country": "Reunion"},
    "TUR": {"country": "Turkey"},
    "COD": {"country": "Democratic Republic of the Congo"},
    "SWZ": {"country": "Eswatini"},
    "FSM": {"country": "Micronesia"},
    "MKD": {"country": "North Macedonia"},
    "MDA": {"country": "Moldova"},
    "PRK": {"country": "North Korea"},
    "KOR": {"country": "South Korea"},
    "TZA": {"country": "Tanzania"},
    "GBR": {"country": "United Kingdom"},
    "USA": {"country": "United States"},
    "IRN": {"country": "Iran"},
    "SYR": {"country": "Syria"},
    "BOL": {"country": "Bolivia"},
    "VEN": {"country": "Venezuela"},
    "RUS": {"country": "Russia"},
    "VNM": {"country": "Vietnam"},
    "LAO": {"country": "Laos"},
    "TWN": {"country": "Taiwan"},
}

for iso3, fixes in manual_fixes.items():
    mask = ember_countries["iso3"] == iso3
    for col, val in fixes.items():
        ember_countries.loc[mask, col] = val

# Confirm Kosovo is now resolved
print("Kosovo row:")
print(ember_countries[ember_countries["iso3"] == "XKX"].to_string(index=False))

# Check for any remaining nulls
remaining_nulls = ember_countries[ember_countries[["iso2", "country"]].isnull().any(axis=1)]
print(f"\nRows still missing iso2 or country: {len(remaining_nulls)}")
if len(remaining_nulls) > 0:
    print(remaining_nulls[["name_ember", "iso3", "iso2", "country"]].to_string(index=False))

Kosovo row:
name_ember iso3 iso2 country
    Kosovo  XKX   XK  Kosovo

Rows still missing iso2 or country: 0


In [8]:
# Show all rows where the Ember name differs from the canonical country name.
# These are the cases where downstream joins on name_ember would have failed
# without the crosswalk. Worth reviewing to confirm all look correct.
name_diffs = ember_countries[ember_countries["name_ember"] != ember_countries["country"]]
print(f"Countries where Ember name differs from canonical name: {len(name_diffs)}")
print(name_diffs[["name_ember", "country", "iso2", "iso3"]].to_string(index=False))

Countries where Ember name differs from canonical name: 29
                                 name_ember                          country iso2 iso3
                              Bahamas (the)                          Bahamas   BS  BHS
                         Bosnia Herzegovina           Bosnia and Herzegovina   BA  BIH
                                 Cabo Verde                       Cape Verde   CV  CPV
                       Cayman Islands (the)                   Cayman Islands   KY  CYM
             Central African Republic (the)         Central African Republic   CF  CAF
                              Comoros (the)                          Comoros   KM  COM
     Congo (the Democratic Republic of the) Democratic Republic of the Congo   CD  COD
                                Congo (the)                            Congo   CG  COG
                         Cook Islands (the)                     Cook Islands   CK  COK
                              Cote d'Ivoire                      Ivory 

In [9]:
# Extract the distinct country names from cbam_defaults and attempt to match
# each one to the crosswalk on the canonical country name first, then on the
# Ember name as a fallback. Anything that doesn't match either way needs a
# manual entry in name_cbam_defaults.
cbam_countries = pd.DataFrame({"name_cbam_defaults": sorted(defaults["country"].unique())})

# Attempt match on canonical name first
cbam_countries = cbam_countries.merge(
    ember_countries[["country", "iso2", "iso3"]],
    left_on="name_cbam_defaults",
    right_on="country",
    how="left"
).drop(columns="country")

unmatched = cbam_countries[cbam_countries["iso2"].isnull()]
print(f"CBAM countries not matched on canonical name: {len(unmatched)}")
print(unmatched["name_cbam_defaults"].to_string(index=False))

CBAM countries not matched on canonical name: 6
                         Brunei
                        Curaçao
                  Côte d'Ivoire
Democratic Republic of the Cong
                  Myanmar_Burma
                        Türkiye


In [16]:
# Map the six unmatched CBAM country names to their correct ISO codes.
# For each, we first attempt to look up the match from ember_countries.
# If the country isn't in Ember (e.g. Curaçao), we fall back to a direct
# manual entry. This approach makes failures visible rather than silent.

cbam_manual = {
    "Brunei":                            {"iso2": "BN", "direct": False},
    "Curaçao":                           {"iso2": "CW", "iso3": "CUW", "direct": True},
    "Côte d'Ivoire":                     {"iso2": "CI", "direct": False},
    "Democratic Republic of the Cong":   {"iso2": "CD", "direct": False},
    "Myanmar_Burma":                     {"iso2": "MM", "direct": False},
    "Türkiye":                           {"iso2": "TR", "direct": False},
}

for cbam_name, config in cbam_manual.items():
    mask = cbam_countries["name_cbam_defaults"] == cbam_name
    if config["direct"]:
        # Country not in Ember — apply ISO codes directly
        cbam_countries.loc[mask, "iso2"] = config["iso2"]
        cbam_countries.loc[mask, "iso3"] = config["iso3"]
        print(f"  Direct entry applied: {cbam_name} -> {config['iso2']} / {config['iso3']}")
    else:
        # Look up from ember_countries via ISO2
        match = ember_countries[ember_countries["iso2"] == config["iso2"]]
        if len(match) == 0:
            print(f"  WARNING: {cbam_name} ({config['iso2']}) not found in Ember — needs manual review")
        else:
            cbam_countries.loc[mask, "iso2"] = match.iloc[0]["iso2"]
            cbam_countries.loc[mask, "iso3"] = match.iloc[0]["iso3"]
            print(f"  Resolved via Ember: {cbam_name} -> {config['iso2']} / {match.iloc[0]['iso3']}")

# Confirm all resolved
still_unmatched = cbam_countries[cbam_countries["iso2"].isnull()]
print(f"CBAM countries still unmatched: {len(still_unmatched)}")
print(f"\nAll CBAM countries resolved: {len(cbam_countries)}")
print(cbam_countries.to_string(index=False))

  Resolved via Ember: Brunei -> BN / BRN
  Direct entry applied: Curaçao -> CW / CUW
  Resolved via Ember: Côte d'Ivoire -> CI / CIV
  Resolved via Ember: Democratic Republic of the Cong -> CD / COD
  Resolved via Ember: Myanmar_Burma -> MM / MMR
  Resolved via Ember: Türkiye -> TR / TUR
CBAM countries still unmatched: 0

All CBAM countries resolved: 119
             name_cbam_defaults iso2 iso3
                        Albania   AL  ALB
                        Algeria   DZ  DZA
                         Angola   AO  AGO
                      Argentina   AR  ARG
                        Armenia   AM  ARM
                      Australia   AU  AUS
                     Azerbaijan   AZ  AZE
                        Bahrain   BH  BHR
                     Bangladesh   BD  BGD
                        Belarus   BY  BLR
                          Benin   BJ  BEN
                        Bolivia   BO  BOL
         Bosnia and Herzegovina   BA  BIH
                         Brazil   BR  BRA
             

In [13]:
# Merge the CBAM country mapping onto the ember_countries base to produce
# the final crosswalk. The canonical 'country' column comes from ember_countries
# (already cleaned in cell 1.3). Curaçao is not in Ember so we add it manually.
# The name_cbam_defaults column preserves the raw source name for auditability.

# Add Curaçao to ember_countries base since it was missing
curacao_row = pd.DataFrame([{
    "name_ember": None,
    "iso3":       "CUW",
    "iso2":       "CW",
    "country":    "Curacao",
}])
ember_base = pd.concat([ember_countries, curacao_row], ignore_index=True)

# Build crosswalk: start from ember_base, left join CBAM names onto it
crosswalk = ember_base.merge(
    cbam_countries[["name_cbam_defaults", "iso2"]],
    on="iso2",
    how="left"
)[["iso2", "iso3", "country", "name_cbam_defaults", "name_ember"]]

print(f"Crosswalk shape: {crosswalk.shape}")
print(f"Countries with a CBAM default name: {crosswalk['name_cbam_defaults'].notna().sum()}")
print(f"Countries without a CBAM default name: {crosswalk['name_cbam_defaults'].isna().sum()}")
print(f"\nSample:")
print(crosswalk.head(10).to_string(index=False))

Crosswalk shape: (216, 5)
Countries with a CBAM default name: 119
Countries without a CBAM default name: 97

Sample:
iso2 iso3             country name_cbam_defaults          name_ember
  AF  AFG         Afghanistan                NaN         Afghanistan
  AL  ALB             Albania            Albania             Albania
  DZ  DZA             Algeria            Algeria             Algeria
  AS  ASM      American Samoa                NaN      American Samoa
  AO  AGO              Angola             Angola              Angola
  AG  ATG Antigua and Barbuda                NaN Antigua and Barbuda
  AR  ARG           Argentina          Argentina           Argentina
  AM  ARM             Armenia            Armenia             Armenia
  AW  ABW               Aruba                NaN               Aruba
  AU  AUS           Australia          Australia           Australia


In [14]:
# Save the completed crosswalk to data/clean/.
# This is the single source of truth for all country identifier joins
# across the project. Any future dataset that needs country standardization
# should join onto this table.
crosswalk.to_csv(clean / "country_crosswalk.csv", index=False)

print(f"Saved: {clean / 'country_crosswalk.csv'}")
print(f"Shape: {crosswalk.shape}")

Saved: ../data/clean/country_crosswalk.csv
Shape: (216, 5)


### 1. Observations

- 216 countries in the crosswalk, seeded from Ember's country list.
- 119 of these have a corresponding CBAM default entry. The remaining 97
  are countries Ember covers that are not in the CBAM defaults — mostly
  small economies or territories the EU does not import CBAM-relevant
  goods from in meaningful volumes.
- Kosovo (XKX) is the only country not resolvable via pycountry — added
  manually with ISO2 XK, consistent with Eurostat convention.
- Curaçao appears in CBAM defaults but not in Ember — added manually
  with ISO2 CW, ISO3 CUW.
- All canonical country names are common English, ASCII only. Source
  variant names are preserved in name_cbam_defaults and name_ember
  for auditability.

## 2. CBAM Defaults 

(`cbam_defaults.csv`)

Cleaning steps:
1. Standardize country names to canonical English via the crosswalk
2. Strip spaces from cn_code
3. Investigate the zero-emission row and the null default_2026
4. Rename production_route to production_route_code, add human-readable
   production_route column using the benchmark annex key
5. Fix the non-breaking space value in production_route_code

In [43]:
# Work on a copy to leave the original dataframe unchanged.
# Join the crosswalk on name_cbam_defaults to replace the raw source
# country name with the canonical English name.
defaults_clean = defaults.copy()

defaults_clean = defaults_clean.merge(
    crosswalk[["name_cbam_defaults", "country"]],
    left_on="country",
    right_on="name_cbam_defaults",
    how="left"
)

# Replace the original country column with the canonical name,
# drop the redundant name_cbam_defaults column from the merge
defaults_clean["country"] = defaults_clean["country_y"]
defaults_clean = defaults_clean.drop(columns=["country_x", "country_y", "name_cbam_defaults"])

# Confirm no country names failed to match the crosswalk
unmatched = defaults_clean[defaults_clean["country"].isnull()]
print(f"Rows with NO crosswalk match: {len(unmatched)}")
if len(unmatched) > 0:
    print(unmatched[["country", "cn_code"]].drop_duplicates().to_string(index=False))

print(f"\nDistinct canonical country names: {defaults_clean['country'].nunique()}")
print(f"\nSample:")
print(defaults_clean[["country", "cn_code", "description"]].head(5).to_string(index=False))

Rows with NO crosswalk match: 0

Distinct canonical country names: 119

Sample:
country    cn_code                     description
Albania 2523 10 00                    Grey clinker
Albania 2523 29 00            Grey Portland cement
Albania 2523 90 00          Grey hydraulic cements
Albania 2808 00 00 Nitric acid; sulphonitric acids
Albania   28142000     Ammonia in aqueous solution


In [44]:
# CN codes are stored as spaced strings (e.g. "2523 10 00").
# Strip spaces to produce a clean code for joining with trade flow data,
# which stores CN codes as integers. The original spaced format is not retained
# since the stripped version is unambiguous and easier to work with.
defaults_clean["cn_code"] = defaults_clean["cn_code"].str.replace(" ", "")

print("CN code length distribution after stripping spaces:")
print(defaults_clean["cn_code"].str.len().value_counts().sort_index())
print("\nSample:")
print(defaults_clean["cn_code"].head(5).to_list())

CN code length distribution after stripping spaces:
cn_code
4    1111
6    1960
8    7600
Name: count, dtype: int64

Sample:
['25231000', '25232900', '25239000', '28080000', '28142000']


In [45]:
# One row was flagged in assessment as having both direct_emissions
# and total_emissions equal to zero. Identify it and decide how to handle it.
zero_row = defaults_clean[defaults_clean["total_emissions"] == 0]
print(f"Zero-emission rows: {len(zero_row)}")
print(zero_row.to_string(index=False))

Zero-emission rows: 1
 cn_code description  direct_emissions  indirect_emissions  total_emissions  default_2026  default_2027  default_2028_onwards production_route country
28041000    Hydrogen            0.0000                 NaN           0.0000        0.0000        0.0000                0.0000              NaN    Mali


In [46]:
# One row was flagged as having a null default_2026 value.
# Identify it — likely a placeholder or dash in the source xlsx.
null_2026 = defaults_clean[defaults_clean["default_2026"].isnull()]
print(f"Rows with null default_2026: {len(null_2026)}")
print(null_2026.to_string(index=False))

Rows with null default_2026: 1
 cn_code                                                                                                                                   description  direct_emissions  indirect_emissions  total_emissions  default_2026  default_2027  default_2028_onwards production_route country
73061100 Line pipe of a kind used for oil or gas pipelines, welded, of flat-rolled products of stainless steel, of an external diameter of <= 406,4 mm            2.9500                 NaN           2.9500           NaN        3.5400                3.8350                    Chile


In [47]:
# Both anomalies identified in assessment are confirmed as source-faithful:
#
# 1. Mali, CN 28041000 (Hydrogen): all emission values and default values
#    are genuinely zero in the source xlsx. Mali has no reported hydrogen
#    production emissions. Retained as-is.
#
# 2. Chile, CN 73061100 (stainless steel line pipe): default_2026 is a dash
#    in both the EU Commission xlsx and in the legally binding source,
#    Commission Implementing Regulation (EU) 2025/2621 (EUR-Lex, page 471/2400).
#    The EU Commission did not publish a 2026 default value for this specific
#    product/origin combination. default_2027 (3.540) and default_2028_onwards
#    (3.835) are present and valid. The null is legally confirmed — do not impute.
#
# No changes made in this cell. Documented for auditability.

print("Mali hydrogen row:")
print(defaults_clean[
    (defaults_clean["country"] == "Mali") &
    (defaults_clean["cn_code"] == "28041000")
].to_string(index=False))

print("\nChile stainless steel pipe row:")
print(defaults_clean[
    (defaults_clean["country"] == "Chile") &
    (defaults_clean["cn_code"] == "73061100")
].to_string(index=False))

Mali hydrogen row:
 cn_code description  direct_emissions  indirect_emissions  total_emissions  default_2026  default_2027  default_2028_onwards production_route country
28041000    Hydrogen            0.0000                 NaN           0.0000        0.0000        0.0000                0.0000              NaN    Mali

Chile stainless steel pipe row:
 cn_code                                                                                                                                   description  direct_emissions  indirect_emissions  total_emissions  default_2026  default_2027  default_2028_onwards production_route country
73061100 Line pipe of a kind used for oil or gas pipelines, welded, of flat-rolled products of stainless steel, of an external diameter of <= 406,4 mm            2.9500                 NaN           2.9500           NaN        3.5400                3.8350                    Chile


In [48]:
# Rename the original production_route column to production_route_code to
# preserve the source letter codes for joining to steel_route_intensity.
# Add a new production_route column with the full human-readable name
# per the benchmark annex key in Commission Implementing Regulation (EU) 2025/2620.
# Combined codes like (C)/(F) get both names written out in full.

defaults_clean = defaults_clean.rename(columns={"production_route": "production_route_code"})

route_name_map = {
    "(A)":    "Grey Clinker / Cement",
    "(B)":    "White Clinker / Cement",
    "(C)":    "Carbon Steel, BF-BOF",
    "(D)":    "Carbon Steel, DRI-EAF",
    "(E)":    "Carbon Steel, Scrap-EAF",
    "(F)":    "Low Alloy Steel, BF-BOF",
    "(G)":    "Low Alloy Steel, DRI-EAF",
    "(H)":    "Low Alloy Steel, Scrap-EAF",
    "(J)":    "High Alloy Steel, EAF",
    "(K)":    "Primary Aluminium",
    "(L)":    "Secondary Aluminium",
    "(C)/(F)":"Carbon Steel, BF-BOF / Low Alloy Steel, BF-BOF",
    "(E)/(H)":"Carbon Steel, Scrap-EAF / Low Alloy Steel, Scrap-EAF",
}

defaults_clean["production_route"] = defaults_clean["production_route_code"].map(route_name_map)

# Confirm the mapping — check distinct combinations of code and name
print("Production route code to name mapping (distinct values):")
print(defaults_clean.groupby(
    ["production_route_code", "production_route"], dropna=False
).size().reset_index(name="rows").to_string(index=False))

Production route code to name mapping (distinct values):
production_route_code                                     production_route  rows
                  (A)                                Grey Clinker / Cement   171
                  (B)                               White Clinker / Cement    32
                  (C)                                 Carbon Steel, BF-BOF  2675
              (C)/(F)       Carbon Steel, BF-BOF / Low Alloy Steel, BF-BOF   130
                  (E)                              Carbon Steel, Scrap-EAF   515
              (E)/(H) Carbon Steel, Scrap-EAF / Low Alloy Steel, Scrap-EAF    25
                  (F)                              Low Alloy Steel, BF-BOF   884
                  (H)                           Low Alloy Steel, Scrap-EAF   170
                  (K)                                    Primary Aluminium   720
                  (L)                                  Secondary Aluminium   864
                                                    

In [ ]:
# The production_route_code column contains \xa0 (non-breaking space) characters
# in cells that appear empty in the source xlsx. This is an Excel artifact where
# cells were cleared with a space rather than deleted, resulting in a whitespace
# string instead of a true null. These are replaced with null here.
print("Rows with non-breaking space in production_route_code before fix:",
      (defaults_clean["production_route_code"] == "\xa0").sum())

defaults_clean["production_route_code"] = defaults_clean["production_route_code"].replace("\xa0", None)

print("Rows with non-breaking space after fix:",
      (defaults_clean["production_route_code"] == "\xa0").sum())
print("\nNull production_route_code rows after fix:",
      defaults_clean["production_route_code"].isnull().sum())

Rows with non-breaking space in production_route_code before fix: 3452
Rows with non-breaking space after fix: 0

Null production_route_code rows after fix: 4485


In [51]:
# Final shape and null check before saving — confirm row count is unchanged
# and all cleaning steps produced the expected result.
print(f"Shape (should match original 10671 rows): {defaults_clean.shape}")
print(f"\nNull counts:")
print(defaults_clean.isnull().sum())
print(f"\nDistinct countries: {defaults_clean['country'].nunique()}")
print(f"\nSample:")
print(defaults_clean.head(3).to_string(index=False))

Shape (should match original 10671 rows): (10671, 11)

Null counts:
cn_code                     0
description                 0
direct_emissions            0
indirect_emissions       7926
total_emissions             0
default_2026                1
default_2027                0
default_2028_onwards        0
production_route_code    4485
country                     0
production_route         4485
dtype: int64

Distinct countries: 119

Sample:
 cn_code            description  direct_emissions  indirect_emissions  total_emissions  default_2026  default_2027  default_2028_onwards production_route_code country      production_route
25231000           Grey clinker            0.8700              0.0000           0.8700        0.9570        1.0440                1.1310                   (A) Albania Grey Clinker / Cement
25232900   Grey Portland cement            0.9000              0.0300           0.9300        1.0230        1.1160                1.2090                   NaN Albania           

In [54]:
# Save the cleaned CBAM defaults to data/clean/.
# Original file in data/processed/ is unchanged.
# Reorder columns to match a logical reading order before saving
defaults_clean = defaults_clean[[
    "country", "cn_code", "description",
    "direct_emissions", "indirect_emissions", "total_emissions",
    "default_2026", "default_2027", "default_2028_onwards",
    "production_route_code", "production_route"
]]
defaults_clean.to_csv(clean / "cbam_defaults_clean.csv", index=False)
print(f"Saved: {clean / 'cbam_defaults_clean.csv'}")
print(f"Shape: {defaults_clean.shape}")

Saved: ../data/clean/cbam_defaults_clean.csv
Shape: (10671, 11)


### 2. Observations

**Country names**
- All 119 country names standardized to canonical common English via the
  crosswalk. Six source names required manual mapping: Brunei, Curaçao,
  Côte d'Ivoire, Democratic Republic of the Cong (truncated), Myanmar_Burma,
  and Türkiye.

**CN codes**
- Spaces stripped from all CN codes. Lengths unchanged: 4-digit (1,111),
  6-digit (1,960), 8-digit (7,600).

**Production route**
- Column renamed from production_route to production_route_code to preserve
  the original benchmark annex letter codes for joining.
- New production_route column added with full human-readable names per the
  benchmark annex in Commission Implementing Regulation (EU) 2025/2620.
- 3,452 non-breaking space (\xa0) values converted to null.

**Documented anomalies — no action taken:**
- Mali, CN 28041000: zero emission values confirmed in source xlsx and in
  Commission Implementing Regulation (EU) 2025/2621. Retained as-is.
- Chile, CN 73061100: null default_2026 confirmed as a dash in both the
  source xlsx and Commission Implementing Regulation (EU) 2025/2621
  (page 471/2400). Legally confirmed absence of a 2026 default value.
  Retained as null.